<a href="https://colab.research.google.com/github/ocalin911/Deep-Learning-Models-of-Mathematical-Physics/blob/Volume-II/Chapter-4-Nonholonomic-Mechanics/Untitled152.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Test for Nonholonomy**

**Direct Frobenius Residual Test for Rolling Wheel Constraints**

We check nonholonomy of the Pfaffian system on $R^4$ with $q=(x,y,\psi,\phi)$:


$$\omega_1 = dx - \cos(\psi) d\phi$$
$$   \omega_2  = dy - \sin(\psi) d\phi.$$

In [ ]:
# ============================================================
# Direct Frobenius Residual Test for Rolling Wheel Constraints
# Ready to run in Google Colab (TensorFlow / Keras)
#
# Pfaffian system on R^4(q) with q=(x,y,psi,phi):
#   ω1 = dx - cos(psi) dphi
#   ω2 = dy - sin(psi) dphi
#
# We test Frobenius integrability via the residual
#   R(q) = sum_{α=1}^2 E_{X,Y in D_q} | dωα(q)(X,Y) |^2
# where D_q = ker ω1(q) ∩ ker ω2(q).
# ============================================================

import tensorflow as tf
import numpy as np

tf.random.set_seed(0)
np.random.seed(0)

# ------------------------------------------------------------
# 1) Define the coefficient matrix A(q) for ωα = sum_j aα^j(q) dq_j
# Coordinate order: (x, y, psi, phi)
# ω1: [1, 0, 0, -cos(psi)]
# ω2: [0, 1, 0, -sin(psi)]
# ------------------------------------------------------------
@tf.function
def A_of_q(q):
    # q: (B,4) tensor
    psi = q[:, 2]
    cospsi = tf.cos(psi)
    sinpsi = tf.sin(psi)

    row1 = tf.stack([tf.ones_like(psi), tf.zeros_like(psi), tf.zeros_like(psi), -cospsi], axis=1)
    row2 = tf.stack([tf.zeros_like(psi), tf.ones_like(psi), tf.zeros_like(psi), -sinpsi], axis=1)
    A = tf.stack([row1, row2], axis=1)  # (B,2,4)
    return A

# ------------------------------------------------------------
# 2) Compute an orthonormal basis for D_q = nullspace(A(q))
# Using SVD per batch element: A = U S V^T
# The last (n-m)=2 columns of V form an orthonormal basis for null(A)
# ------------------------------------------------------------
@tf.function
def nullspace_basis(A):
    # A: (B,2,4)
    # returns X_basis: (B, 4, 2) whose columns are basis vectors in R^4
    # i.e. X_basis[...,i] is X_i
    s, u, v = tf.linalg.svd(A, full_matrices=True)  # v: (B,4,4) = V
    # Nullspace dimension is 2 (since m=2, n=4) for generic q
    X_basis = v[:, :, 2:4]  # (B,4,2)
    return X_basis

# ------------------------------------------------------------
# 3) dωα evaluation using coordinate formula from aα^j(q):
# ωα = aα^j dq_j, so
# dωα = sum_{j<k} (∂_j aα^k - ∂_k aα^j) dq_j ∧ dq_k
# and for vectors X,Y:
# dωα(X,Y) = sum_{j<k} (∂_j aα^k - ∂_k aα^j) (X^j Y^k - X^k Y^j)
#
# Here, only dependence is on psi, so derivatives are sparse.
# We'll implement general autodiff for clarity.
# ------------------------------------------------------------
@tf.function
def a_coeffs(q):
    # Returns a: (B, m=2, n=4) coefficients aα^j(q)
    psi = q[:, 2]
    cospsi = tf.cos(psi)
    sinpsi = tf.sin(psi)

    a1 = tf.stack([tf.ones_like(psi), tf.zeros_like(psi), tf.zeros_like(psi), -cospsi], axis=1)  # (B,4)
    a2 = tf.stack([tf.zeros_like(psi), tf.ones_like(psi), tf.zeros_like(psi), -sinpsi], axis=1)  # (B,4)
    a = tf.stack([a1, a2], axis=1)  # (B,2,4)
    return a

@tf.function
def domega_on_XY(q, X, Y):
    """
    q: (B,4)
    X,Y: (B,4) vectors in D_q
    Returns vals: (B,2) where vals[:,α] = dωα(q)(X,Y)
    """
    with tf.GradientTape(persistent=True) as tape:
        tape.watch(q)
        a = a_coeffs(q)  # (B,2,4)

    # Jacobian: J[b, α, j, k] = ∂ a[b, α, k] / ∂ q[b, j]
    J = tape.batch_jacobian(a, q)  # (B,2,4,4)
    del tape

    # Compute antisymmetric coefficients Cα_{j,k} = ∂_j aα^k - ∂_k aα^j
    # We'll compute dω(X,Y) = sum_{j<k} C_{j,k} (X^j Y^k - X^k Y^j)
    # Vectorized approach:
    # C: (B,2,4,4)
    C = J - tf.transpose(J, perm=[0, 1, 3, 2])

    # W_{j,k} = X^j Y^k - X^k Y^j  => (B,4,4)
    X_col = tf.expand_dims(X, axis=2)  # (B,4,1)
    Y_row = tf.expand_dims(Y, axis=1)  # (B,1,4)
    XY = X_col * Y_row                  # (B,4,4)
    Y_col = tf.expand_dims(Y, axis=2)
    X_row = tf.expand_dims(X, axis=1)
    YX = Y_col * X_row                  # (B,4,4)
    W = XY - YX                         # (B,4,4)

    # dωα(X,Y) = 1/2 * sum_{j,k} Cα_{j,k} W_{j,k}
    # (factor 1/2 because both j<k and k<j are included in full sum)
    vals = 0.5 * tf.reduce_sum(C * tf.expand_dims(W, axis=1), axis=[2, 3])  # (B,2)
    return vals

# ------------------------------------------------------------
# 4) Sample random X,Y in D_q from nullspace basis
# Let basis columns be X1,X2; choose random Gaussian coeffs and normalize.
# ------------------------------------------------------------
@tf.function
def sample_XY_from_D(X_basis):
    # X_basis: (B,4,2)
    B = tf.shape(X_basis)[0]
    coeffX = tf.random.normal((B, 2))
    coeffY = tf.random.normal((B, 2))

    X = tf.einsum('bij,bj->bi', X_basis, coeffX)  # (B,4)
    Y = tf.einsum('bij,bj->bi', X_basis, coeffY)  # (B,4)

    # Normalize to unit length (optional, but helps scale stability)
    X = X / (tf.norm(X, axis=1, keepdims=True) + 1e-12)
    Y = Y / (tf.norm(Y, axis=1, keepdims=True) + 1e-12)
    return X, Y

# ------------------------------------------------------------
# 5) Compute Frobenius residual R(q) via Monte Carlo
# R(q) ≈ mean over K samples of sum_{α} |dωα(X,Y)|^2
# ------------------------------------------------------------
@tf.function
def frobenius_residual(q, K=16):
    # q: (B,4)
    A = A_of_q(q)
    X_basis = nullspace_basis(A)

    accum = tf.zeros((tf.shape(q)[0],), dtype=q.dtype)
    for _ in tf.range(K):
        X, Y = sample_XY_from_D(X_basis)
        vals = domega_on_XY(q, X, Y)  # (B,2)
        accum += tf.reduce_sum(tf.square(vals), axis=1)
    return accum / tf.cast(K, q.dtype)  # (B,)

# ------------------------------------------------------------
# 6) Run experiment: sample points q and report statistics
# Note: ω's depend only on psi, so x,y,phi can be arbitrary.
# We'll sample psi uniformly in [-pi, pi].
# ------------------------------------------------------------
def run_test(num_points=5000, K=32):
    # Sample q = (x,y,psi,phi)
    x = np.random.uniform(-1.0, 1.0, size=(num_points, 1)).astype(np.float32)
    y = np.random.uniform(-1.0, 1.0, size=(num_points, 1)).astype(np.float32)
    psi = np.random.uniform(-np.pi, np.pi, size=(num_points, 1)).astype(np.float32)
    phi = np.random.uniform(-1.0, 1.0, size=(num_points, 1)).astype(np.float32)
    q = np.concatenate([x, y, psi, phi], axis=1)

    q_tf = tf.convert_to_tensor(q)

    R = frobenius_residual(q_tf, K=tf.constant(K))
    R_np = R.numpy()

    print("Direct Frobenius residual test for rolling wheel constraints")
    print(f"Samples: {num_points}, Monte Carlo pairs per q: {K}")
    print(f"Residual stats: min={R_np.min():.6e}, mean={R_np.mean():.6e}, median={np.median(R_np):.6e}, max={R_np.max():.6e}")
    # Fraction near zero (tolerance)
    tol = 1e-6
    frac_small = np.mean(R_np < tol)
    print(f"Fraction with R(q) < {tol}: {frac_small:.4f}")
    print("\nInterpretation:")
    print("If the Pfaffian system were integrable (holonomic), R(q) would be ~0 for all q.")
    print("Here R(q) is generically positive, indicating Frobenius condition fails -> nonholonomic constraints.")

    return q, R_np

# Run
q_samples, R_values = run_test(num_points=5000, K=32)


Direct Frobenius residual test for rolling wheel constraints
Samples: 5000, Monte Carlo pairs per q: 32
Residual stats: min=1.412136e-01, mean=2.499302e-01, median=2.502407e-01, max=3.510111e-01
Fraction with R(q) < 1e-06: 0.0000

Interpretation:
If the Pfaffian system were integrable (holonomic), R(q) would be ~0 for all q.
Here R(q) is generically positive, indicating Frobenius condition fails -> nonholonomic constraints.


**Direct Frobenius Residual Test for a Pfaffian system on $R^3$**

In [ ]:
# ============================================================
# Direct Frobenius Residual Test for a Pfaffian system on R^3
# Google Colab / TensorFlow-Keras compatible
#
# Pfaffian system:
#   ω1 = dz - y dx
#   ω2 = dy
#
# This system is holonomic. The Frobenius residual should vanish
# (up to numerical noise).
# ============================================================

import tensorflow as tf
import numpy as np

# Reproducibility
tf.random.set_seed(0)
np.random.seed(0)

# ------------------------------------------------------------
# 1) Coefficient matrix A(q) for ωα = aα^j dq_j
# Coordinates: q = (x, y, z)
# ------------------------------------------------------------
@tf.function
def A_of_q(q):
    # q: (B,3)
    y = q[:, 1]
    row1 = tf.stack([-y, tf.zeros_like(y), tf.ones_like(y)], axis=1)  # ω1
    row2 = tf.stack([tf.zeros_like(y), tf.ones_like(y), tf.zeros_like(y)], axis=1)  # ω2
    return tf.stack([row1, row2], axis=1)  # (B,2,3)

@tf.function
def a_coeffs(q):
    y = q[:, 1]
    a1 = tf.stack([-y, tf.zeros_like(y), tf.ones_like(y)], axis=1)
    a2 = tf.stack([tf.zeros_like(y), tf.ones_like(y), tf.zeros_like(y)], axis=1)
    return tf.stack([a1, a2], axis=1)  # (B,2,3)

# ------------------------------------------------------------
# 2) Nullspace basis of D_q = ker(A(q))
# Since n=3, m=2 => dim(D)=1
# ------------------------------------------------------------
@tf.function
def nullspace_basis(A):
    _, _, v = tf.linalg.svd(A, full_matrices=True)  # v: (B,3,3)
    return v[:, :, 2:3]  # (B,3,1)

# ------------------------------------------------------------
# 3) Compute dωα(X,Y) via autodiff
# ------------------------------------------------------------
@tf.function
def domega_on_XY(q, X, Y):
    with tf.GradientTape(persistent=True) as tape:
        tape.watch(q)
        a = a_coeffs(q)

    # Jacobian J[b,α,j,k] = ∂ aα^k / ∂ q_j
    J = tape.batch_jacobian(a, q)
    del tape

    # Antisymmetric coefficients
    C = J - tf.transpose(J, perm=[0, 1, 3, 2])

    # W_{jk} = X^j Y^k - X^k Y^j
    Xc = tf.expand_dims(X, axis=2)
    Yr = tf.expand_dims(Y, axis=1)
    Yc = tf.expand_dims(Y, axis=2)
    Xr = tf.expand_dims(X, axis=1)
    W = Xc * Yr - Yc * Xr

    # dωα(X,Y)
    return 0.5 * tf.reduce_sum(C * tf.expand_dims(W, axis=1), axis=[2, 3])

# ------------------------------------------------------------
# 4) Sample X,Y in D_q
# Since dim(D)=1, all X,Y are colinear => dωα(X,Y)=0 identically
# ------------------------------------------------------------
@tf.function
def sample_XY_from_D(X_basis):
    B = tf.shape(X_basis)[0]
    coeffX = tf.random.normal((B, 1))
    coeffY = tf.random.normal((B, 1))

    X = tf.einsum('bij,bj->bi', X_basis, coeffX)
    Y = tf.einsum('bij,bj->bi', X_basis, coeffY)

    X = X / (tf.norm(X, axis=1, keepdims=True) + 1e-12)
    Y = Y / (tf.norm(Y, axis=1, keepdims=True) + 1e-12)
    return X, Y

# ------------------------------------------------------------
# 5) Frobenius residual
# ------------------------------------------------------------
@tf.function
def frobenius_residual(q, K=16):
    A = A_of_q(q)
    X_basis = nullspace_basis(A)

    acc = tf.zeros((tf.shape(q)[0],), dtype=q.dtype)
    for _ in tf.range(K):
        X, Y = sample_XY_from_D(X_basis)
        vals = domega_on_XY(q, X, Y)
        acc += tf.reduce_sum(tf.square(vals), axis=1)

    return acc / tf.cast(K, q.dtype)

# ------------------------------------------------------------
# 6) Run experiment
# ------------------------------------------------------------
def run_test(num_points=3000, K=16):
    q = np.random.uniform(-1.0, 1.0, size=(num_points, 3)).astype(np.float32)
    q_tf = tf.convert_to_tensor(q)

    R = frobenius_residual(q_tf, K=tf.constant(K)).numpy()

    print("Direct Frobenius residual test for ω1 = dz - y dx, ω2 = dy")
    print(f"Samples: {num_points}, Monte Carlo pairs per q: {K}")
    print(f"Residual statistics:")
    print(f"  min   = {R.min():.3e}")
    print(f"  mean  = {R.mean():.3e}")
    print(f"  median= {np.median(R):.3e}")
    print(f"  max   = {R.max():.3e}")

    tol = 1e-8
    print(f"Fraction with R(q) < {tol}: {np.mean(R < tol):.4f}")
    print("\nInterpretation:")
    print("Since dim(D)=1, dωα(X,Y)=0 for all X,Y in D by skew-symmetry.")
    print("The near-zero residual confirms that the Pfaffian system is holonomic.")

    return q, R

# Run
q_samples, R_values = run_test()


Direct Frobenius residual test for ω1 = dz - y dx, ω2 = dy
Samples: 3000, Monte Carlo pairs per q: 16
Residual statistics:
  min   = 0.000e+00
  mean  = 0.000e+00
  median= 0.000e+00
  max   = 0.000e+00
Fraction with R(q) < 1e-08: 1.0000

Interpretation:
Since dim(D)=1, dωα(X,Y)=0 for all X,Y in D by skew-symmetry.
The near-zero residual confirms that the Pfaffian system is holonomic.
